# Tam Veri Seti (IBL Brainwidemap) iSTTC vs. Binned ACF & 2×2×2 Varyans Analizi
Bu notebook:
1. Gerekli kütüphaneleri kurar ve isteğe bağlı olarak Google Drive'ı bağlar (böylece Colab bağlantısı kopsa bile kaldığı probe'dan devam eder).
2. Düzeltilmiş `isttc_pipeline.py` ve tüm veri setini tarayan `variance.py` dosyalarını yan yana oluşturur.
3. IBL sunucusundan **tüm Brainwidemap probe kayıtlarını** çekip her nöron için hem Binned ACF hem iSTTC hesaplar, her probe bittiğinde `dual_estimator_tau.csv` dosyasına kaydeder ve en sonunda **2×2×2 matrisini (`eight_cell_table.csv`)** üretir.

In [ ]:
!pip install -q ONE-api iblatlas pymc arviz pandas numpy scipy matplotlib seaborn tqdm numba joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.5 MB/s eta 0:00:00


In [ ]:
# İSTEĞE BAĞLI: Verilerin Colab kapanınca silinmemesi için Google Drive'ı bağlayın
import os
USE_GOOGLE_DRIVE = True  # Drive kullanmak istemiyorsanız False yapın
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/IBL_iSTTC_Full_Analysis'
    os.makedirs(WORK_DIR, exist_ok=True)
    os.chdir(WORK_DIR)
    print('Çalışma klasörü:', os.getcwd())


In [ ]:
%%writefile isttc_pipeline.py
# -*- coding: utf-8 -*-
"""
isttc_pipeline.py
------------------
iSTTC intrinsic timescale analysis pipeline (auto-fetches IBL Brainwidemap sessions).

Requirements:
    pip install ONE-api iblatlas pandas numpy scipy matplotlib seaborn tqdm numba joblib

Usage:
    python isttc_pipeline.py
"""

# -- Imports ------------------------------------------------------------------
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats
from numba import njit
from tqdm import tqdm
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')

# ==============================================================================
# PATHS & CONSTANTS -- edit these to match your local setup
# ==============================================================================
OUTPUT_CSV              = 'good_isttc.csv'
REGISTRY_CSV            = 'ibl_master_pid_registry.csv'
OUTPUT_DIR              = 'isttc_summary_plots'
IBL_GOOD_LABEL          = 1  # IBL standard: label == 1 -> all 3 QC metrics passed
SPONTANEOUS_EPOCH_LABEL = 'passivePeriods.spontaneousActivity'
MAX_SESSIONS            = None  # Set to an int (e.g. 10) for a quick test, or None for ALL sessions


# ==============================================================================
# CORE iSTTC FUNCTIONS
# ==============================================================================

@njit(cache=True)
def _tiling_proportion_nb(spike_times, start, end, delta_t, total_duration, shift):
    """T_A with slice bounds and shift - no array allocation."""
    n = end - start
    if n == 0:
        return 0.0
    cum_max_end = spike_times[start] - shift + delta_t
    if cum_max_end > total_duration:
        cum_max_end = total_duration
    s0 = spike_times[start] - shift - delta_t
    if s0 < 0.0:
        s0 = 0.0
    total = cum_max_end - s0
    for i in range(start + 1, end):
        s = spike_times[i] - shift - delta_t
        if s < 0.0:
            s = 0.0
        e = spike_times[i] - shift + delta_t
        if e > total_duration:
            e = total_duration
        overlap = cum_max_end - s
        if overlap < 0.0:
            overlap = 0.0
        total += (e - s) - overlap
        if e > cum_max_end:
            cum_max_end = e
    if total < 0.0:
        total = 0.0
    if total > total_duration:
        total = total_duration
    return total / total_duration


@njit(cache=True)
def _spikes_in_tiling_nb(spike_times_test, t_start, t_end, shift_test,
                        spike_times_ref,  r_start, r_end, shift_ref,
                        delta_t):
    """P_A|B: fraction of (test-shift_test) spikes within delta_t of (ref-shift_ref) spikes."""
    n_test = t_end - t_start
    if n_test == 0 or (r_end - r_start) == 0:
        return 0.0
    count = 0
    for i in range(t_start, t_end):
        val = spike_times_test[i] - shift_test
        lo = val - delta_t + shift_ref
        hi = val + delta_t + shift_ref
        left = r_start
        right = r_end
        while left < right:
            mid = (left + right) // 2
            if spike_times_ref[mid] < lo:
                left = mid + 1
            else:
                right = mid
        l2 = r_start
        r2 = r_end
        while l2 < r2:
            mid = (l2 + r2) // 2
            if spike_times_ref[mid] <= hi:
                l2 = mid + 1
            else:
                r2 = mid
        if l2 > left:
            count += 1
    return count / n_test


@njit(cache=True)
def _isttc_nb(spike_times, total_duration, delta_t, nlags, lag_shift):
    """iSTTC lag loop - fully compiled, zero array allocation inside loop."""
    isttc_values = np.full(nlags, np.nan)
    isttc_values[0] = 1.0
    n = len(spike_times)
    si = 0
    ei = n
    for i in range(1, nlags):
        lag = i * lag_shift
        dur = total_duration - lag
        while si < n and spike_times[si] < lag:
            si += 1
        while ei > 0 and spike_times[ei - 1] > total_duration - lag:
            ei -= 1
        if (n - si) < 2 or ei < 2:
            continue
        TA = _tiling_proportion_nb(spike_times, si, n,  delta_t, dur, lag)
        TB = _tiling_proportion_nb(spike_times, 0,  ei, delta_t, dur, 0.0)
        PA = _spikes_in_tiling_nb(spike_times, si, n,  lag,
                                  spike_times, 0,  ei, 0.0, delta_t)
        PB = _spikes_in_tiling_nb(spike_times, 0,  ei, 0.0,
                                  spike_times, si, n,  lag, delta_t)
        denom1 = 1.0 - PA * TB
        denom2 = 1.0 - PB * TA
        if abs(denom1) < 1e-10 or abs(denom2) < 1e-10:
            continue
        isttc_values[i] = 0.5 * ((PA - TB) / denom1 + (PB - TA) / denom2)
    return isttc_values


def calculate_isttc(spike_times, total_duration,
                    delta_t=0.005, nlags=600, lag_shift=0.01):
    """iSTTC autocorrelation function. (Pochinok et al. 2026, Sec. 7.4)

    Parameters are in SECONDS (`delta_t`, `lag_shift`, `total_duration`).
    Returns `(lags_s, isttc_values)` where `lags_s` is in SECONDS.
    """
    if len(spike_times) == 0:
        return np.array([]), np.array([])
    spike_times = np.ascontiguousarray(np.sort(spike_times), dtype=np.float64)
    lags = np.arange(nlags) * lag_shift
    isttc_values = _isttc_nb(spike_times, float(total_duration),
                             float(delta_t), int(nlags), float(lag_shift))
    return lags, isttc_values


# ==============================================================================
# MULTI-EXPONENTIAL FITTING & MODEL SELECTION
# ==============================================================================

def _multi_exp(t, *params):
    """Sum of M exponentials: sum_j c_j * exp(-t / tau_j)."""
    val = np.zeros_like(t, dtype=float)
    for j in range(0, len(params), 2):
        c, tau = params[j], params[j+1]
        val += c * np.exp(-t / tau)
    return val


def fit_multi_exponential(lags, isttc_values, start_lag_idx=1,
                          max_components=4, min_component_fraction=0.01):
    """
    1 to 4-component exponential fit + BIC model selection.
    Expects `lags` in SECONDS.
    Returns timescales in MILLISECONDS (`tau_effective_ms`, `tau_1_ms`..`tau_4_ms`,
    along with legacy aliases `tau_effective`, `tau_1`..`tau_4`), `c_1`..`c_4`,
    `bic_values`, `r_squared`, and 95% CI (`ci_lower_ms`, `ci_upper_ms`).
    """
    valid = ~np.isnan(isttc_values)
    lags_fit = lags[valid][start_lag_idx:]
    vals_fit = isttc_values[valid][start_lag_idx:]

    empty = {
        'n_timescales': np.nan,
        'tau_1_ms': np.nan, 'tau_2_ms': np.nan, 'tau_3_ms': np.nan, 'tau_4_ms': np.nan,
        'tau_1':    np.nan, 'tau_2':    np.nan, 'tau_3':    np.nan, 'tau_4':    np.nan,
        'c_1':      np.nan, 'c_2':      np.nan, 'c_3':      np.nan, 'c_4':      np.nan,
        'tau_effective_ms': np.nan,
        'tau_effective':    np.nan,
        'r_squared': np.nan, 'bic_values': {},
        'fit_values': np.full_like(lags, np.nan),
        'ci_lower_ms': np.nan, 'ci_upper_ms': np.nan, 'ci_excludes_zero': False,
        'tau_at_bound': False,
    }

    if len(lags_fit) < 4:
        return empty

    n_data = len(lags_fit)
    best_bic, best_result, bic_values = np.inf, None, {}

    tau_lower = 0.005
    tau_upper = max(float(lags_fit[-1]), 0.5)

    for M in range(1, max_components + 1):
        n_params = 2 * M
        if n_data <= n_params + 1:
            break

        amp_init = max(np.max(np.abs(vals_fit)) / M, 1e-6)
        tau_init = np.geomspace(tau_lower, tau_upper, M)
        p0 = sum([[amp_init, tau_init[j]] for j in range(M)], [])
        lb = sum([[0.0, tau_lower] for _ in range(M)], [])
        ub = sum([[np.inf, tau_upper] for _ in range(M)], [])

        try:
            popt, pcov = curve_fit(_multi_exp, lags_fit, vals_fit,
                                  p0=p0, bounds=(lb, ub),
                                  maxfev=20000, method='trf')
        except Exception:
            continue

        c_vals, tau_vals = popt[0::2], popt[1::2]
        c_sum = np.sum(c_vals)
        if c_sum <= 0 or np.any(c_vals / c_sum < min_component_fraction):
            continue
        if np.any(c_vals > 10.0):
            continue

        fitted = _multi_exp(lags_fit, *popt)
        ss_res = np.sum((vals_fit - fitted) ** 2)
        ss_tot = np.sum((vals_fit - np.mean(vals_fit)) ** 2)
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        bic = n_data * np.log(ss_res / n_data + 1e-30) + n_params * np.log(n_data)
        bic_values[M] = bic

        if bic < best_bic:
            best_bic = bic
            tau_eff_s = float(np.sum(c_vals * tau_vals) / c_sum)

            try:
                grad = sum([[(tau_vals[j] - tau_eff_s) / c_sum,
                             c_vals[j] / c_sum] for j in range(M)], [])
                grad = np.array(grad)
                std_tau_eff = np.sqrt(max(grad @ pcov @ grad, 0.0))
                t_crit = stats.t.ppf(0.975, max(n_data - n_params, 1))
                ci_lo = (tau_eff_s - t_crit * std_tau_eff) * 1000.0
                ci_hi = (tau_eff_s + t_crit * std_tau_eff) * 1000.0
                ci_ex0 = bool(ci_lo > 0)
            except Exception:
                ci_lo, ci_hi, ci_ex0 = np.nan, np.nan, bool(tau_eff_s > 0)

            sort_idx      = np.argsort(tau_vals)
            tau_sorted_ms = tau_vals[sort_idx] * 1000.0
            c_sorted      = c_vals[sort_idx]
            tau_at_bound  = bool(np.any(tau_vals >= tau_upper * 0.99))

            t1 = float(tau_sorted_ms[0]) if M >= 1 else np.nan
            t2 = float(tau_sorted_ms[1]) if M >= 2 else np.nan
            t3 = float(tau_sorted_ms[2]) if M >= 3 else np.nan
            t4 = float(tau_sorted_ms[3]) if M >= 4 else np.nan
            tau_eff_ms = tau_eff_s * 1000.0

            best_result = {
                'n_timescales': M,
                'tau_1_ms': t1, 'tau_2_ms': t2, 'tau_3_ms': t3, 'tau_4_ms': t4,
                'tau_1':    t1, 'tau_2':    t2, 'tau_3':    t3, 'tau_4':    t4,
                'c_1': float(c_sorted[0]) if M >= 1 else np.nan,
                'c_2': float(c_sorted[1]) if M >= 2 else np.nan,
                'c_3': float(c_sorted[2]) if M >= 3 else np.nan,
                'c_4': float(c_sorted[3]) if M >= 4 else np.nan,
                'tau_effective_ms': tau_eff_ms,
                'tau_effective':    tau_eff_ms,
                'r_squared': float(r2),
                'bic_values': bic_values,
                'fit_values': _multi_exp(lags, *popt),
                'ci_lower_ms': float(ci_lo),
                'ci_upper_ms': float(ci_hi),
                'ci_excludes_zero': ci_ex0,
                'tau_at_bound': tau_at_bound,
            }

    if best_result is None:
        return empty

    best_result['bic_values'] = bic_values
    return best_result


# ==============================================================================
# AUXILIARY METRICS & QUALITY CRITERIA
# ==============================================================================

def compute_local_variance(spike_times):
    """Local Variance (Lv) - Shinomoto et al. 2009."""
    if len(spike_times) < 3:
        return np.nan
    isi = np.diff(np.sort(spike_times))
    if len(isi) < 2:
        return np.nan
    return float(3.0 * np.mean(
        ((isi[:-1] - isi[1:]) / (isi[:-1] + isi[1:] + 1e-30)) ** 2
    ))


MIN_SPIKE_COUNT = 100

def check_quality_criteria(lags, isttc_values, fit_results, spike_count=None):
    q = {
        'acf_decline_50_200ms': False,
        'ci_excludes_zero': bool(fit_results.get('ci_excludes_zero', False)),
        'r_squared_above_0p5': False,
        'spike_count_above_min': False,
        'tau_at_bound': bool(fit_results.get('tau_at_bound', False)),
        'passed_quality_filter': False,
    }

    mask = (lags >= 0.05) & (lags <= 0.20)
    valid_vals = isttc_values[mask][~np.isnan(isttc_values[mask])]
    if len(valid_vals) >= 2:
        q['acf_decline_50_200ms'] = bool(valid_vals[0] > valid_vals[-1])

    r2 = fit_results.get('r_squared', np.nan)
    if not np.isnan(r2):
        q['r_squared_above_0p5'] = bool(r2 >= 0.5)

    if spike_count is not None:
        q['spike_count_above_min'] = bool(spike_count >= MIN_SPIKE_COUNT)

    q['passed_quality_filter'] = (
        q['acf_decline_50_200ms'] and
        q['ci_excludes_zero'] and
        q['r_squared_above_0p5'] and
        q['spike_count_above_min'] and
        not q['tau_at_bound']
    )
    return q


# ==============================================================================
# SINGLE NEURON ANALYSIS
# ==============================================================================

def analyze_single_neuron(spike_times, recording_duration,
                          neuron_id='Neuron',
                          delta_t=0.005, nlags=600, lag_shift=0.01,
                          max_components=4):
    spike_times = np.sort(np.asarray(spike_times, dtype=float))
    firing_rate = len(spike_times) / recording_duration if recording_duration > 0 else np.nan
    spike_count = len(spike_times)
    lv          = compute_local_variance(spike_times)

    lags, isttc_values = calculate_isttc(
        spike_times, recording_duration,
        delta_t=delta_t, nlags=nlags, lag_shift=lag_shift
    )
    fit     = fit_multi_exponential(lags, isttc_values, max_components=max_components)
    quality = check_quality_criteria(lags, isttc_values, fit, spike_count=spike_count)

    return {
        'neuron_id':             neuron_id,
        'n_spikes':              spike_count,
        'firing_rate_hz':        firing_rate,
        'local_variance_lv':     lv,
        'n_timescales':          fit['n_timescales'],
        'tau_1_ms':              fit['tau_1_ms'],
        'tau_2_ms':              fit['tau_2_ms'],
        'tau_3_ms':              fit['tau_3_ms'],
        'tau_4_ms':              fit['tau_4_ms'],
        'c_1': fit['c_1'], 'c_2': fit['c_2'],
        'c_3': fit['c_3'], 'c_4': fit['c_4'],
        'tau_effective_ms':      fit['tau_effective_ms'],
        'r_squared':             fit['r_squared'],
        'ci_lower_ms':           fit['ci_lower_ms'],
        'ci_upper_ms':           fit['ci_upper_ms'],
        'bic_1': fit['bic_values'].get(1, np.nan),
        'bic_2': fit['bic_values'].get(2, np.nan),
        'bic_3': fit['bic_values'].get(3, np.nan),
        'bic_4': fit['bic_values'].get(4, np.nan),
        **quality,
        '_lags':          lags,
        '_isttc_values': isttc_values,
        '_fit_values':   fit['fit_values'],
    }


# ==============================================================================
# SESSION HELPERS
# ==============================================================================

def get_spontaneous_window(one, eid, skip_first_minutes=0.0, verbose=True):
    """Find the passive spontaneousActivity epoch for an IBL session.

    In IBL ALF format, `_ibl_passivePeriods.intervalsTable.csv` is loaded either:
      - directly as a pandas DataFrame via `one.load_dataset(eid, '*passivePeriods*', collection='alf')`, or
      - as `{'intervalsTable': DataFrame}` via `one.load_object(eid, 'passivePeriods', collection='alf')`.
    This function handles both structures and NEVER loads active task `trials`.
    """
    passive = None
    try:
        passive = one.load_dataset(eid, '*passivePeriods*', collection='alf')
    except Exception:
        try:
            passive = one.load_object(eid, 'passivePeriods', collection='alf')
        except Exception as exc:
            if verbose:
                print(f"  No passivePeriods found for {eid}: {exc}")
            return None, None, None

    if passive is None:
        if verbose:
            print(f"  No passivePeriods data returned for {eid}.")
        return None, None, None

    if isinstance(passive, dict) and 'intervalsTable' in passive:
        table = passive['intervalsTable']
    else:
        table = passive

    if 'spontaneousActivity' not in table:
        if verbose:
            keys = list(table.keys()) if hasattr(table, 'keys') else type(table).__name__
            print(f"  No 'spontaneousActivity' in passivePeriods (found: {keys}) for {eid}.")
        return None, None, None

    sp_interval = np.asarray(table['spontaneousActivity'], dtype=float).reshape(-1)
    if sp_interval.size < 2:
        if verbose:
            print(f"  Invalid spontaneousActivity shape {sp_interval.shape} for {eid}.")
        return None, None, None

    t_start_raw = float(sp_interval[0])
    t_end       = float(sp_interval[1])
    t_start     = t_start_raw + (skip_first_minutes * 60.0)

    if np.isnan(t_start) or np.isnan(t_end) or t_start >= t_end:
        if verbose:
            print(f"  Invalid spontaneousActivity window [{t_start}, {t_end}] for {eid}.")
        return None, None, None

    if verbose:
        print(f"  Epoch: {SPONTANEOUS_EPOCH_LABEL} [{t_start:.1f}s, {t_end:.1f}s]")
    return t_start, t_end, SPONTANEOUS_EPOCH_LABEL


def _get_qc_labels(clusters):
    """Extract IBL QC labels from `clusters` (1 == good unit passing all 3 IBL QC criteria)."""
    if clusters is None:
        return None
    if 'label' in clusters:
        return np.asarray(clusters['label'], dtype=float)
    if 'metrics' in clusters and 'label' in getattr(clusters['metrics'], 'columns', []):
        return np.asarray(clusters['metrics']['label'], dtype=float)
    return None


def _get_cluster_regions(one, eid, probe, clusters):
    """Assign Allen CCF (`brain_region`) and Beryl (`beryl_region`) acronyms per cluster if available."""
    n_clusters = len(clusters['channels']) if clusters is not None and 'channels' in clusters else 0
    allen_regions = np.full(n_clusters, np.nan, dtype=object)
    beryl_regions = np.full(n_clusters, np.nan, dtype=object)
    if n_clusters == 0:
        return allen_regions, beryl_regions

    try:
        channels = one.load_object(eid, 'channels', collection=f'alf/{probe}/pykilosort')
    except Exception:
        try:
            channels = one.load_object(eid, 'channels', collection=f'alf/{probe}')
        except Exception:
            return allen_regions, beryl_regions

    clu_channels = np.asarray(clusters['channels'], dtype=int)
    if 'acronym' in channels:
        allen_regions = np.asarray(channels['acronym'], dtype=object)[clu_channels]
    elif 'brainLocationAcronyms_ccf_2017' in channels:
        allen_regions = np.asarray(channels['brainLocationAcronyms_ccf_2017'], dtype=object)[clu_channels]

    if 'brainLocationIds_ccf_2017' in channels:
        try:
            from iblatlas.atlas import BrainRegions
            br = BrainRegions()
            chan_ids = np.asarray(channels['brainLocationIds_ccf_2017'])
            beryl_regions = br.id2acronym(chan_ids, mapping='Beryl')[clu_channels]
        except Exception:
            pass
    return allen_regions, beryl_regions


def _process_cluster(cluster_id, spike_dict, recording_duration,
                     qc_labels, apply_ibl_qc, min_spikes,
                     eid, probe, epoch_source, t_start, t_end,
                     allen_region=np.nan, beryl_region=np.nan):
    """Worker function for parallel cluster processing."""
    if apply_ibl_qc:
        if qc_labels is None or qc_labels[cluster_id] != IBL_GOOD_LABEL:
            return None

    spike_times = spike_dict.get(cluster_id)
    if spike_times is None or len(spike_times) < min_spikes:
        return None

    result = analyze_single_neuron(
        spike_times,
        recording_duration=recording_duration,
        neuron_id=f'Cluster_{cluster_id}',
    )

    return {
        'session_id':           eid,
        'probe':                probe,
        'cluster_id':           int(cluster_id),
        'epoch_source':         epoch_source,
        't_start_s':            t_start,
        't_end_s':              t_end,
        'brain_region':         allen_region,
        'beryl_region':         beryl_region,
        'ap_um':                np.nan,
        'dv_um':                np.nan,
        'ml_um':                np.nan,
        'n_spikes_spontaneous': result['n_spikes'],
        'firing_rate_hz':        result['firing_rate_hz'],
        'local_variance_lv':     result['local_variance_lv'],
        'n_timescales':          result['n_timescales'],
        'tau_1_ms':              result['tau_1_ms'],
        'tau_2_ms':              result['tau_2_ms'],
        'tau_3_ms':              result['tau_3_ms'],
        'tau_4_ms':              result['tau_4_ms'],
        'c_1': result['c_1'], 'c_2': result['c_2'],
        'c_3': result['c_3'], 'c_4': result['c_4'],
        'tau_effective_ms':      result['tau_effective_ms'],
        'r_squared':             result['r_squared'],
        'ci_lower_ms':           result['ci_lower_ms'],
        'ci_upper_ms':           result['ci_upper_ms'],
        'bic_1': result['bic_1'], 'bic_2': result['bic_2'],
        'bic_3': result['bic_3'], 'bic_4': result['bic_4'],
        'acf_decline_50_200ms':  result['acf_decline_50_200ms'],
        'ci_excludes_zero':     result['ci_excludes_zero'],
        'r_squared_above_0p5':   result['r_squared_above_0p5'],
        'spike_count_above_min': result['spike_count_above_min'],
        'tau_at_bound':          result['tau_at_bound'],
        'passed_quality_filter': result['passed_quality_filter'],
    }


# ==============================================================================
# SESSION ANALYSIS
# ==============================================================================

def analyze_session(eid, one, probe='probe00',
                    min_spikes=100,
                    apply_ibl_qc=True,
                    n_jobs=-1,
                    verbose=True,
                    preloaded=None):

    if verbose:
        print(f"\n{'='*65}")
        print(f"  iSTTC | Session: {eid} | Probe: {probe}")
        print(f"{'='*65}")

    if preloaded is not None:
        t_start      = preloaded['t_start']
        t_end        = preloaded['t_end']
        epoch_source = preloaded['epoch_source']
        spikes       = preloaded['spikes']
        clusters     = preloaded['clusters']
    else:
        t_start, t_end, epoch_source = get_spontaneous_window(one, eid, verbose=verbose)
        if t_start is None:
            print(f"  SKIP: No passive epoch found for {eid}.")
            return pd.DataFrame()
        try:
            spikes = one.load_object(eid, 'spikes',
                                     collection=f'alf/{probe}/pykilosort')
        except Exception as e:
            print(f"  Could not load spike data: {e}")
            return pd.DataFrame()
        try:
            clusters = one.load_object(eid, 'clusters',
                                       collection=f'alf/{probe}/pykilosort')
        except Exception:
            clusters = None

    recording_duration = t_end - t_start

    if verbose:
        print(f"  Epoch source      : {epoch_source}")
        print(f"  Duration          : {recording_duration:.1f}s ({recording_duration/60:.1f} min)")

    qc_labels = None
    if apply_ibl_qc:
        qc_labels = _get_qc_labels(clusters)
        if qc_labels is None:
            raise RuntimeError(
                f"No QC 'label' found in clusters (nor in clusters['metrics']) for {eid} / {probe}. "
                "Refusing to continue without IBL QC."
            )

    allen_regions, beryl_regions = _get_cluster_regions(one, eid, probe, clusters)

    spike_times_all    = np.array(spikes['times'])
    spike_clusters_all = np.array(spikes['clusters'])
    sp_mask            = (spike_times_all >= t_start) & (spike_times_all <= t_end)
    spont_times        = spike_times_all[sp_mask] - t_start
    spont_clusters     = spike_clusters_all[sp_mask]
    unique_clusters    = np.unique(spont_clusters)

    if verbose:
        print(f"  Spikes in SP window : {sp_mask.sum():,}")
        print(f"  Clusters in SP window: {len(unique_clusters):,}")

    spike_dict = {cid: spont_times[spont_clusters == cid] for cid in unique_clusters}

    raw_results = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(_process_cluster)(
            cid, spike_dict,
            recording_duration, qc_labels, apply_ibl_qc, min_spikes,
            eid, probe, epoch_source, t_start, t_end,
            allen_regions[cid] if cid < len(allen_regions) else np.nan,
            beryl_regions[cid] if cid < len(beryl_regions) else np.nan,
        )
        for cid in tqdm(unique_clusters, desc='  Clusters', unit='cluster',
                        ncols=80, leave=True)
    )

    results_list = [r for r in raw_results if r is not None]
    df = pd.DataFrame(results_list)

    if verbose and len(df) > 0:
        n_pass = df['passed_quality_filter'].sum()
        med_tau = df.loc[df['passed_quality_filter'], 'tau_effective_ms'].median()
        print(f"\n  Completed!")
        print(f"  Neurons analysed  : {len(df)}")
        print(f"  Passed QC         : {n_pass} ({100*n_pass/len(df):.1f}%)")
        print(f"  Median τ_eff      : {med_tau:.1f} ms")
        print(f"{'='*65}\n")

    return df


# ==============================================================================
# BATCH RUNNER & AUTO-DISCOVERY OF IBL BRAINWIDEMAP SESSIONS
# ==============================================================================

def fetch_all_brainwidemap_sessions(one, registry_csv=REGISTRY_CSV):
    """Load `(eid, pid, probe_name)` tuples from `registry_csv` if present,
    or automatically query all IBL Brainwidemap insertions from Alyx.
    """
    if os.path.exists(registry_csv):
        reg = pd.read_csv(registry_csv)
        print(f"Loaded {len(reg)} probes from local {registry_csv}.")
        return list(zip(reg['eid'], reg['pid'], reg['probe_name']))

    print("Fetching Brainwidemap insertions directly from IBL Alyx server...")
    insertions = one.alyx.rest("insertions", "list", no_cache=True, tag="Brainwidemap")
    rows = []
    for ins in insertions:
        probe_name = ins.get("name", "")
        if not str(probe_name).startswith("probe"):
            continue
        session = ins.get("session", {})
        eid = session.get("id") if isinstance(session, dict) else session
        subject = session.get("subject") if isinstance(session, dict) else None
        lab = session.get("lab") if isinstance(session, dict) else None
        rows.append({
            "pid": ins.get("id"),
            "eid": eid,
            "probe_name": probe_name,
            "subject": subject,
            "lab": lab,
        })
    df = pd.DataFrame(rows).drop_duplicates(subset=["pid"]).reset_index(drop=True)
    df.to_csv(registry_csv, index=False)
    print(f"Saved {len(df)} Brainwidemap probes to {registry_csv}.")
    return list(zip(df['eid'], df['pid'], df['probe_name']))


def run_sessions(one, pids, output_csv=OUTPUT_CSV, verbose=True):
    """
    Run iSTTC analysis on a list of (eid, pid, probe_name) tuples.
    Automatically skips PIDs already present in `output_csv` so interrupted runs can resume.
    """
    done_pids = set()
    if os.path.exists(output_csv) and os.path.getsize(output_csv) > 0:
        try:
            existing_df = pd.read_csv(output_csv, usecols=['pid'])
            done_pids = set(existing_df['pid'].dropna().astype(str))
            if done_pids:
                print(f"Resuming: skipping {len(done_pids)} PIDs already saved in {output_csv}.")
        except Exception:
            pass

    all_results = []
    skipped     = []

    for eid, pid, probe_name in pids:
        if str(pid) in done_pids:
            continue
        try:
            df = analyze_session(eid, one, probe_name, verbose=verbose)
            if len(df) > 0:
                df = df.copy()
                df['pid'] = pid
                all_results.append(df)

                qc_mask = (
                    df['passed_quality_filter']
                    .astype(str).str.strip().str.lower().eq('true')
                )
                good_session = df[qc_mask].copy()

                if len(good_session) > 0:
                    try:
                        if os.path.exists(output_csv) and os.path.getsize(output_csv) > 0:
                            existing = pd.read_csv(output_csv)
                            pd.concat([existing, good_session], ignore_index=True)\
                              .to_csv(output_csv, index=False)
                            print(f"  Saved {len(good_session)} neurons from {pid} "
                                  f"-- running total: {len(existing) + len(good_session)}")
                        else:
                            good_session.to_csv(output_csv, index=False)
                            print(f"  Saved {len(good_session)} neurons from {pid} to new file.")
                    except Exception as save_err:
                        print(f"  SAVE ERROR -- PID {pid}: {save_err}")
                else:
                    print(f"  PID {pid}: 0 neurons passed QC -- nothing saved.")
            else:
                skipped.append(pid)

        except Exception as e:
            print(f"  ERROR -- PID {pid} / session {eid}: {e}")
            skipped.append(pid)
            continue

    if skipped:
        print(f"\n  Skipped PIDs ({len(skipped)}): {skipped}")

    if not all_results:
        print("No new results collected in this run.")
        return pd.DataFrame()

    combined  = pd.concat(all_results, ignore_index=True)
    qc_mask   = (
        combined['passed_quality_filter']
        .astype(str).str.strip().str.lower().eq('true')
    )
    qc_good   = combined[qc_mask]
    n_total   = len(combined)
    n_qc      = len(qc_good)
    median_tau = qc_good['tau_effective_ms'].median() if n_qc > 0 else float('nan')

    print(f"\n{'='*65}")
    print("  BATCH COMPLETE")
    print(f"  Sessions processed : {len(all_results)} / {len(pids)}")
    print(f"  Total neurons      : {n_total}")
    print(f"  Passed QC          : {n_qc} ({100 * n_qc / n_total:.1f}%)")
    print(f"  Median τ_eff       : {median_tau:.1f} ms")
    print(f"  Output file        : {output_csv}")
    print(f"{'='*65}\n")

    return qc_good


# ==============================================================================
# VISUALIZATION
# ==============================================================================

def plot_single_neuron(result, title=None):
    lags_ms = result['_lags'] * 1000
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.plot(lags_ms, result['_isttc_values'], 'o-', color='#2E86AB',
            markersize=3, linewidth=1.8, label='iSTTC', alpha=0.8)
    if not np.all(np.isnan(result['_fit_values'])):
        tau_eff = result['tau_effective_ms']
        n_tau   = result['n_timescales']
        r2      = result['r_squared']
        lbl = (f'{n_tau}-exp fit | τ_eff={tau_eff:.1f}ms | R²={r2:.3f}'
               if not np.isnan(tau_eff) else 'Fit')
        ax.plot(lags_ms, result['_fit_values'], 'r-', linewidth=2.2,
                label=lbl, alpha=0.85)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.35)
    ax.axvspan(50, 200, alpha=0.07, color='orange', label='50-200ms QC')
    ax.set_xlabel('Lag (ms)', fontsize=12, fontweight='bold')
    ax.set_ylabel('iSTTC', fontsize=12, fontweight='bold')
    ax.set_title(title or result['neuron_id'], fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25, linestyle=':')
    ax.spines[['top', 'right']].set_visible(False)

    ax = axes[1]
    ax.axis('off')
    qc = {k: ('✓' if result.get(k) else '✗') for k in
          ['acf_decline_50_200ms', 'ci_excludes_zero',
           'r_squared_above_0p5', 'spike_count_above_min']}
    comp = ""
    for k in range(1, 5):
        tk = result.get(f'tau_{k}_ms', np.nan)
        ck = result.get(f'c_{k}', np.nan)
        if tk is not None and not np.isnan(float(tk)):
            comp += f"  τ{k}={tk:.1f}ms (c{k}={ck:.3f})\n"
    passed = '✓ PASSED' if result['passed_quality_filter'] else '✗ FAILED'
    text = (f"Neuron: {result['neuron_id']}\n"
            f"Spikes: {result['n_spikes']:,} | FR: {result['firing_rate_hz']:.2f}Hz | "
            f"Lv: {result['local_variance_lv']:.3f}\n\n"
            f"--- {result['n_timescales']} component(s) ---\n{comp}"
            f"τ_eff = {result['tau_effective_ms']:.2f} ms\n"
            f"CI: [{result['ci_lower_ms']:.1f}, {result['ci_upper_ms']:.1f}] ms\n"
            f"R² = {result['r_squared']:.4f}\n\n"
            f"[{qc['acf_decline_50_200ms']}] ACF decline (50-200ms)\n"
            f"[{qc['ci_excludes_zero']}] CI > 0\n"
            f"[{qc['r_squared_above_0p5']}] R² >= 0.5\n"
            f"[{qc['spike_count_above_min']}] Spike count\n"
            f"OVERALL: {passed}")
    ax.text(0.05, 0.95, text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', family='monospace',
            bbox=dict(boxstyle='round', facecolor='#eaf4fb', alpha=0.5))
    plt.tight_layout()
    return fig


# ==============================================================================
# ENTRY POINT -- automatically fetches all Brainwidemap sessions if none listed
# ==============================================================================

if __name__ == '__main__':
    from one.api import ONE

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    one = ONE(
        base_url='https://openalyx.internationalbrainlab.org',
        username='intbrainlab',
        password='international',
        silent=True,
    )

    SESSIONS = fetch_all_brainwidemap_sessions(one, registry_csv=REGISTRY_CSV)
    if MAX_SESSIONS is not None:
        SESSIONS = SESSIONS[:MAX_SESSIONS]

    print(f"Starting iSTTC pipeline on {len(SESSIONS)} probes...")
    results = run_sessions(one, pids=SESSIONS, output_csv=OUTPUT_CSV)

    if len(results) > 0:
        print(f"\nDone. {len(results)} QC-passing neurons saved to {OUTPUT_CSV}")

Writing isttc_pipeline.py


In [ ]:
%%writefile variance.py
# -*- coding: utf-8 -*-
"""
Estimator x Model x Scale disentangling analysis (FULL IBL DATASET VERSION)
===========================================================================
Runs dual estimation (Binned ACF + iSTTC on the identical 6 s lag grid) across
ALL IBL Brainwidemap probes (with incremental saving, resume support, and
automatic disk-cache cleanup), then builds the common neuron set and computes
the 2 x 2 x 2 = 8-cell matrix (`eight_cell_table.csv`).

Three factors, 2 x 2 x 2 = 8 cells:
    ESTIMATOR : binned ACF        vs  iSTTC
    MODEL     : ordinary R^2      vs  hierarchical variance share
    SCALE     : raw tau           vs  log10 tau

Usage:
    python variance.py                 # fetch/resume ALL IBL sessions + build 2x2x2 matrix
    python variance.py --reuse         # skip downloading; build 2x2x2 matrix from existing dual_estimator_tau.csv
    python variance.py --max-probes 20 # process up to 20 probes, then build 2x2x2 matrix
"""

import os
import sys
import shutil

import numpy as np
import pandas as pd
from numba import njit
from joblib import Parallel, delayed
from tqdm import tqdm

from isttc_pipeline import (
    calculate_isttc,
    fit_multi_exponential,
    get_spontaneous_window,
    fetch_all_brainwidemap_sessions,
)

# =============================================================================
# GLOBAL SETTINGS (one place, so both estimators always share them)
# =============================================================================

LAG_STEP_S = 0.010          # lag spacing of the fit grid (iSTTC lag_shift)
N_LAGS = 600                # -> fit window = 600 x 10 ms = 6 s for BOTH estimators
ISTTC_DELTA_T_S = 0.005     # iSTTC synchronicity window
ACF_BIN_S = 0.005           # binned-ACF bin size (lag grid is a multiple of it)
MIN_SPIKES = 100
MIN_R2_FIT = 0.5
MIN_NEURONS_PER_REGION = 15
IBL_GOOD_LABEL = 1          # IBL standard: label == 1 -> all QC metrics passed
REQUIRE_PASSIVE = True
PASSIVE_EPOCH_LABELS = ("passivePeriods.spontaneousActivity", "spontaneousActivity")

CSV_PATH = "dual_estimator_tau.csv"
REGISTRY_CSV = "ibl_master_pid_registry.csv"
CLEAN_CACHE_AFTER_PROBE = True  # Prevents Colab disk from filling up across 500+ probes


# =============================================================================
# PART 1: BINNED ACF (Shi et al. 2025, Eq. 1)
# =============================================================================

@njit(cache=True)
def _binned_acf_nb(counts, nlags):
    n = len(counts)
    mean_all = 0.0
    for i in range(n):
        mean_all += counts[i]
    mean_all /= n

    var = 0.0
    for i in range(n):
        d = counts[i] - mean_all
        var += d * d
    var /= (n - 1)

    ac = np.full(nlags, np.nan)
    if var == 0.0:
        return ac

    for j in range(nlags):
        if j >= n:
            break
        m = n - j
        if m < 2:
            break
        mu1 = 0.0
        for i in range(m):
            mu1 += counts[i]
        mu1 /= m
        mu2 = 0.0
        for i in range(j, n):
            mu2 += counts[i]
        mu2 /= m

        s = 0.0
        for i in range(m):
            s += (counts[i] - mu1) * (counts[i + j] - mu2)
        ac[j] = s / (var * m)

    return ac


def calculate_binned_acf_on_grid(spike_times, total_duration, target_lags_s,
                                 bin_size=ACF_BIN_S):
    """
    Binned ACF evaluated EXACTLY at `target_lags_s` (seconds), i.e. the lag
    grid that iSTTC used. Returns (lags_s, ac). Returns empty arrays if the
    recording is too short to support the full window (no silent shortening).
    """
    empty = (np.array([]), np.array([]))
    target = np.asarray(target_lags_s, dtype=float)
    if len(spike_times) == 0 or len(target) == 0:
        return empty

    idx = np.rint(target / bin_size).astype(int)
    if not np.allclose(idx * bin_size, target, atol=1e-9):
        raise ValueError("iSTTC lag grid is not a multiple of the ACF bin size.")

    nlags = int(idx.max()) + 1
    n_bins = int(total_duration / bin_size)
    if n_bins < nlags + 5:
        return empty

    counts, _ = np.histogram(spike_times, bins=n_bins, range=(0, n_bins * bin_size))
    ac = _binned_acf_nb(counts.astype(np.float64), nlags)
    return target, ac[idx]


# =============================================================================
# PART 2: PER-NEURON DUAL ESTIMATION
# =============================================================================

_UNIT_CHECK_DONE = False


def _lags_to_seconds(lags):
    """Ensure lag vector is in seconds. Expected max ~ 6 s (6000 ms)."""
    global _UNIT_CHECK_DONE
    lags = np.asarray(lags, dtype=float)
    if len(lags) and lags.max() > 100:
        lags = lags / 1000.0
        if not _UNIT_CHECK_DONE:
            print("  [units] calculate_isttc returned lags in ms -> converted to s")
    elif not _UNIT_CHECK_DONE:
        print("  [units] calculate_isttc returned lags in s")
    _UNIT_CHECK_DONE = True
    return lags


def _fit_to_tau_ms(lags_s, vals):
    """Fit on a lag vector in SECONDS; return (tau_effective_ms, r2) or (nan, nan)."""
    fit = fit_multi_exponential(lags_s, vals)
    r2 = fit.get("r_squared", np.nan)
    tau_ms = fit.get("tau_effective_ms", fit.get("tau_effective", np.nan))
    if r2 is None or tau_ms is None or np.isnan(r2) or np.isnan(tau_ms):
        return np.nan, np.nan
    if r2 < MIN_R2_FIT:
        return np.nan, np.nan
    return float(tau_ms), float(r2)


def process_neuron_dual(spike_times, total_duration):
    """Both estimators, same spike train, same lag grid, same fit routine."""
    out = {"tau_isttc_ms": np.nan, "tau_binned_ms": np.nan,
           "r2_isttc": np.nan, "r2_binned": np.nan,
           "n_spikes": len(spike_times)}

    if len(spike_times) < MIN_SPIKES:
        return out

    # --- iSTTC ---
    lags_i, vals_i = calculate_isttc(spike_times, total_duration,
                                     delta_t=ISTTC_DELTA_T_S, nlags=N_LAGS,
                                     lag_shift=LAG_STEP_S)
    lags_i = _lags_to_seconds(lags_i)
    vals_i = np.asarray(vals_i, dtype=float)
    if len(lags_i) < N_LAGS:      # full window required
        return out

    out["tau_isttc_ms"], out["r2_isttc"] = _fit_to_tau_ms(lags_i, vals_i)

    # --- binned ACF, on the identical lag grid ---
    lags_b, vals_b = calculate_binned_acf_on_grid(spike_times, total_duration, lags_i)
    if len(lags_b) == N_LAGS and not np.isnan(vals_b).any():
        out["tau_binned_ms"], out["r2_binned"] = _fit_to_tau_ms(lags_b, vals_b)

    return out


def get_beryl_regions_for_clusters(clusters, channels):
    from iblatlas.atlas import BrainRegions

    br = BrainRegions()
    chan_ids = np.asarray(channels["brainLocationIds_ccf_2017"])
    clu_channels = np.asarray(clusters["channels"])
    beryl_acronyms_per_channel = br.id2acronym(chan_ids, mapping="Beryl")
    return beryl_acronyms_per_channel[clu_channels]


def _get_qc_labels(clusters, n_clusters):
    """IBL QC labels. No silent fallback: raise if they cannot be found."""
    labels = None
    if "label" in clusters:
        labels = np.asarray(clusters["label"], dtype=float)
    elif "metrics" in clusters and "label" in getattr(clusters["metrics"], "columns", []):
        labels = np.asarray(clusters["metrics"]["label"], dtype=float)
    if labels is None:
        raise RuntimeError("No QC 'label' found in clusters (nor in clusters['metrics']). "
                           "Refusing to continue without QC.")
    if len(labels) != n_clusters:
        raise RuntimeError(f"QC label length {len(labels)} != n_clusters {n_clusters}")
    return labels


def _load_subject_map(registry_csv):
    """Return dict mapping pid -> subject from registry_csv."""
    if not os.path.exists(registry_csv):
        return {}
    reg = pd.read_csv(registry_csv)
    if "pid" not in reg.columns or "subject" not in reg.columns:
        return {}
    reg["subject"] = reg["subject"].astype(str).str.replace('KS_091', 'KS091', regex=False).str.strip()
    return dict(zip(reg["pid"].astype(str), reg["subject"]))


def run_dual_estimation(one, pids, verbose=True, n_jobs=-1, apply_ibl_qc=True,
                        registry_csv=REGISTRY_CSV, csv_path=CSV_PATH):
    """
    Runs dual estimation across `pids` [(eid, pid, probe_name), ...].
    Saves incrementally to `csv_path` after EVERY probe and automatically resumes
    by skipping PIDs already saved in `csv_path`.
    """
    subject_map = _load_subject_map(registry_csv)

    done_pids = set()
    if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
        try:
            existing_df = pd.read_csv(csv_path, usecols=["pid"])
            done_pids = set(existing_df["pid"].dropna().astype(str))
            print(f"Resuming: {len(done_pids)} probes already saved in {csv_path} will be skipped.")
        except Exception:
            pass

    total_probes = len(pids)
    for idx_probe, (eid, pid, probe_name) in enumerate(pids, start=1):
        if str(pid) in done_pids:
            continue

        if verbose:
            print(f"\n{'=' * 65}")
            print(f"  [{idx_probe}/{total_probes}] Dual estimator | Session: {eid} | Probe: {probe_name}")
            print(f"{'=' * 65}")

        # ---- 1. Epoch window ----
        t_start, t_end, epoch_source = get_spontaneous_window(one, eid, verbose=verbose)
        if t_start is None:
            print(f"  SKIP: no spontaneous/passive epoch for {eid}.")
            continue

        if REQUIRE_PASSIVE and str(epoch_source) not in PASSIVE_EPOCH_LABELS:
            print(f"  SKIP: epoch source is '{epoch_source}', expected passive spontaneousActivity ({eid}).")
            continue

        recording_duration = t_end - t_start
        if recording_duration < N_LAGS * LAG_STEP_S * 2:
            print(f"  SKIP: window {recording_duration:.1f} s too short for the 6 s fit window.")
            continue

        # ---- 2. Load ----
        spikes = clusters = channels = None
        for coll in (f'alf/{probe_name}/pykilosort', f'alf/{probe_name}'):
            try:
                spikes = one.load_object(eid, 'spikes', collection=coll)
                clusters = one.load_object(eid, 'clusters', collection=coll)
                break
            except Exception:
                continue
        for ch_coll in (f'alf/{probe_name}/pykilosort', f'alf/{probe_name}', 'alf'):
            try:
                channels = one.load_object(eid, 'channels', collection=ch_coll)
                if channels is not None and "brainLocationIds_ccf_2017" in channels:
                    break
            except Exception:
                continue

        if spikes is None or clusters is None or channels is None:
            print(f"  SKIP: could not load spikes/clusters/channels for {eid} / {probe_name}")
            continue

        # ---- 3. Regions ----
        try:
            cluster_regions = get_beryl_regions_for_clusters(clusters, channels)
        except Exception as e:
            print(f"  SKIP: could not assign regions: {e}")
            continue

        # ---- 4. QC ----
        n_clusters = len(clusters['channels'])
        try:
            if apply_ibl_qc:
                qc_labels = _get_qc_labels(clusters, n_clusters)
                vals, cnts = np.unique(qc_labels, return_counts=True)
                if verbose:
                    print(f"  QC label distribution: {dict(zip(vals.round(3), cnts))}")
                good = qc_labels == IBL_GOOD_LABEL
                if verbose:
                    print(f"  Good (label == {IBL_GOOD_LABEL}): {good.sum()} / {n_clusters}")
            else:
                good = np.ones(n_clusters, dtype=bool)
        except Exception as e:
            print(f"  SKIP: QC label error: {e}")
            continue

        # ---- 5. Spikes in window ----
        spike_times_all = np.asarray(spikes['times'])
        spike_clusters_all = np.asarray(spikes['clusters'])
        sp_mask = (spike_times_all >= t_start) & (spike_times_all <= t_end)
        spont_times = spike_times_all[sp_mask] - t_start
        spont_clusters = spike_clusters_all[sp_mask]

        cluster_ids = np.arange(n_clusters)
        spike_dict = {cid: spont_times[spont_clusters == cid] for cid in cluster_ids}

        # Resolve subject name from registry or ONE session path
        subj = subject_map.get(str(pid), None)
        if not subj or subj == "nan":
            try:
                session_path = str(one.eid2path(eid))
                if "Subjects/" in session_path:
                    subj = session_path.split("Subjects/")[1].split("/")[0]
            except Exception:
                subj = np.nan

        # ---- 6. Both estimators per good cluster ----
        def _one_cluster(cid):
            if not good[cid]:
                return None
            st = spike_dict.get(cid)
            if st is None or len(st) < MIN_SPIKES:
                return None
            res = process_neuron_dual(st, recording_duration)
            if np.isnan(res["tau_isttc_ms"]) and np.isnan(res["tau_binned_ms"]):
                return None
            res.update({
                "session_id": eid,
                "probe": probe_name,
                "pid": pid,
                "cluster_id": int(cid),
                "region": cluster_regions[cid],
                "epoch_source": str(epoch_source),
                "subject": subj,
            })
            return res

        results = Parallel(n_jobs=n_jobs, prefer="threads")(
            delayed(_one_cluster)(cid) for cid in tqdm(cluster_ids, desc="  Neurons", ncols=80)
        )
        results = [r for r in results if r is not None]
        if verbose:
            print(f"  Neurons with >=1 valid estimate: {len(results)}")

        # ---- 7. Incremental save after each probe ----
        if results:
            probe_df = pd.DataFrame(results)
            write_header = not (os.path.exists(csv_path) and os.path.getsize(csv_path) > 0)
            probe_df.to_csv(csv_path, mode="a", header=write_header, index=False)
            done_pids.add(str(pid))
            if verbose:
                print(f"  -> Appended {len(probe_df)} neurons to {csv_path}")

        # ---- 8. Free local ONE spike cache for this session so disk never fills up ----
        if CLEAN_CACHE_AFTER_PROBE:
            try:
                sess_dir = one.eid2path(eid)
                if sess_dir and os.path.exists(sess_dir):
                    shutil.rmtree(sess_dir, ignore_errors=True)
            except Exception:
                pass

    if not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        print("No usable neurons collected across any session.")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    print(f"\nTotal neurons available in {csv_path}: {len(df)}")
    return df


# =============================================================================
# PART 3: COMMON NEURON SET (built ONCE, used by all 8 cells)
# =============================================================================

TAU_COLS = {"binned_ACF": "tau_binned_ms", "iSTTC": "tau_isttc_ms"}


def build_common_set(df, region_col="region", subject_col="subject",
                     session_col="session_id", pid_col="pid",
                     min_neurons_per_region=MIN_NEURONS_PER_REGION):
    needed = list(TAU_COLS.values()) + [region_col, subject_col, session_col, pid_col]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    d = df.dropna(subset=needed).copy()
    d = d[(d["tau_binned_ms"] > 0) & (d["tau_isttc_ms"] > 0)]

    counts = d[region_col].value_counts()
    keep_regions = counts[counts >= min_neurons_per_region].index
    d = d[d[region_col].isin(keep_regions)].copy()

    for c in [subject_col, session_col, pid_col, region_col]:
        d[c] = d[c].astype(str).astype("category")

    print("\n" + "=" * 70)
    print("COMMON NEURON SET (identical for all 8 cells)")
    print("=" * 70)
    print(f"  input rows                     : {len(df)}")
    print(f"  with BOTH estimators valid     : {len(d)}")
    print(f"  regions (>= {min_neurons_per_region} neurons)      : {d[region_col].nunique()}")
    print(f"  subjects / sessions / probes   : "
          f"{d[subject_col].nunique()} / {d[session_col].nunique()} / {d[pid_col].nunique()}")
    if len(d) < 50 or d[region_col].nunique() < 2:
        raise RuntimeError("Common set too small for the analysis.")
    return d


# =============================================================================
# PART 4: THE TWO MODELS
# =============================================================================

def _transform(d, tau_col, scale):
    if scale == "log":
        return np.log10(d[tau_col].to_numpy(dtype=float))
    if scale == "raw":
        return d[tau_col].to_numpy(dtype=float)
    raise ValueError(scale)


def ordinary_r2(d, tau_col, scale, region_col="region"):
    """Region-only dummy regression R^2 on the common set."""
    from sklearn.linear_model import LinearRegression

    y = _transform(d, tau_col, scale)
    X = pd.get_dummies(d[region_col], drop_first=True).to_numpy(dtype=float)
    y_hat = LinearRegression().fit(X, y).predict(X)
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan


def _find_sd_var(posterior, factor):
    """Bambi names group-level SDs '1|<factor>_sigma'."""
    exact = f"1|{factor}_sigma"
    if exact in posterior.data_vars:
        return exact
    cands = [v for v in posterior.data_vars
             if factor in v and v.lower().endswith("sigma")]
    if len(cands) != 1:
        raise KeyError(f"Cannot identify SD variable for '{factor}': {cands}")
    return cands[0]


def hierarchical_variance_share(d, tau_col, scale, region_col="region",
                                subject_col="subject", session_col="session_id",
                                pid_col="pid", draws=1000, tune=1000, chains=4):
    """
    Variance share per component, computed PER POSTERIOR DRAW
    (var_k / sum_j var_j for each draw), then averaged over draws.
    """
    import bambi as bmb

    data = d[[region_col, subject_col, session_col, pid_col]].copy()
    data["y"] = _transform(d, tau_col, scale)

    formula = (f"y ~ 1 + (1|{region_col}) + (1|{subject_col})"
               f" + (1|{session_col}) + (1|{pid_col})")
    model = bmb.Model(formula, data=data, family="gaussian")
    idata = model.fit(draws=draws, tune=tune, chains=chains, cores=chains,
                      target_accept=0.9, random_seed=42)
    post = idata.posterior

    factors = [region_col, subject_col, session_col, pid_col]
    var = {}
    for f in factors:
        var[f] = post[_find_sd_var(post, f)].values.reshape(-1) ** 2
    var["residual"] = post["sigma"].values.reshape(-1) ** 2

    names = list(var)
    mat = np.stack([var[k] for k in names])            # (components, draws)
    share_draws = mat / mat.sum(axis=0, keepdims=True) # per-draw proportions

    out = {k: float(share_draws[i].mean()) for i, k in enumerate(names)}
    lo, hi = np.percentile(share_draws[0], [3, 97])     # region interval
    out["region_ci_lo"], out["region_ci_hi"] = float(lo), float(hi)
    out["region_share"] = out[region_col]
    return out


# =============================================================================
# PART 5: 2 x 2 x 2 TABLE (estimator x model x scale)
# =============================================================================

def build_8cell_table(d, region_col="region", subject_col="subject",
                      session_col="session_id", pid_col="pid", out_csv="eight_cell_table.csv"):
    rows = []
    n_ref = len(d)
    k_ref = d[region_col].nunique()

    for estimator, tau_col in TAU_COLS.items():
        for scale in ["raw", "log"]:
            # --- ordinary R^2 ---
            r2 = ordinary_r2(d, tau_col, scale, region_col)
            rows.append(dict(estimator=estimator, model="ordinary_R2", scale=scale,
                             value=r2, ci_lo=np.nan, ci_hi=np.nan,
                             n_neurons=len(d), n_regions=d[region_col].nunique()))

            # --- hierarchical variance share ---
            print(f"\n>>> hierarchical fit: {estimator} / {scale}")
            h = hierarchical_variance_share(d, tau_col, scale, region_col,
                                            subject_col, session_col, pid_col)
            rows.append(dict(estimator=estimator, model="hierarchical_share", scale=scale,
                             value=h["region_share"], ci_lo=h["region_ci_lo"],
                             ci_hi=h["region_ci_hi"],
                             n_neurons=len(d), n_regions=d[region_col].nunique()))

    res = pd.DataFrame(rows)

    # same-neuron guarantee
    assert (res["n_neurons"] == n_ref).all() and (res["n_regions"] == k_ref).all(), \
        "Neuron/region sets differ across cells!"

    res.to_csv(out_csv, index=False)

    print("\n" + "=" * 78)
    print(f"REGION EFFECT, estimator x model x scale   (n = {n_ref} neurons, {k_ref} regions, all cells)")
    print("=" * 78)
    print(f"{'':32s} {'binned ACF':>20s} {'iSTTC':>20s}")
    for model in ["ordinary_R2", "hierarchical_share"]:
        for scale in ["raw", "log"]:
            vals = []
            for est in TAU_COLS:
                v = res[(res.estimator == est) & (res.model == model) & (res.scale == scale)]["value"].iloc[0]
                vals.append(v * 100)
            print(f"{model + ' (' + scale + ')':32s} {vals[0]:>19.1f}% {vals[1]:>19.1f}%")
    print("=" * 78)
    print("Reading guide:")
    print("  - columns (fixed row)      -> ESTIMATOR effect")
    print("  - ordinary vs hierarchical -> MODEL effect (fixed estimator + scale)")
    print("  - raw vs log               -> SCALE effect (fixed estimator + model)")
    print(f"Saved {out_csv}")
    return res


# =============================================================================
# ENTRY POINT (FULL IBL DATASET -> 2x2x2 MATRIX)
# =============================================================================

def main():
    max_probes = None
    if "--max-probes" in sys.argv:
        idx = sys.argv.index("--max-probes")
        if idx + 1 < len(sys.argv):
            max_probes = int(sys.argv[idx + 1])

    if "--reuse" in sys.argv and os.path.exists(CSV_PATH):
        df = pd.read_csv(CSV_PATH)
        print(f"Reusing existing {CSV_PATH} ({len(df)} neurons)")
    else:
        from one.api import ONE
        one = ONE(base_url='https://openalyx.internationalbrainlab.org',
                  username='intbrainlab', password='international', silent=True)

        # Fetch ALL Brainwidemap probes automatically (instead of only 8 hardcoded PIDs)
        all_pids = fetch_all_brainwidemap_sessions(one, registry_csv=REGISTRY_CSV)
        if max_probes is not None:
            all_pids = all_pids[:max_probes]
        print(f"Running dual estimation across {len(all_pids)} IBL Brainwidemap probes...")
        df = run_dual_estimation(one, all_pids)

    if len(df) == 0:
        return

    d = build_common_set(df)
    build_8cell_table(d)


if __name__ == "__main__":
    main()


Writing variance.py


In [ ]:
# Tüm veri setini analiz et ve ardından 2x2x2 matrisini oluştur:
!python variance.py


Fetching Brainwidemap insertions directly from IBL Alyx server...
Saved 699 Brainwidemap probes to ibl_master_pid_registry.csv.
Running dual estimation across 699 IBL Brainwidemap probes...

  [1/699] Dual estimator | Session: ebce500b-c530-47de-8cb1-963c552703ea | Probe: probe00
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/churchlandlab_ucla/Subjects/MFD_09/2023-10-19/001/alf/_ibl_passivePeriods.intervalsTable.csv: 100% 207/207 [00:00<00:00, 1.06kB/s]
  Epoch: passivePeriods.spontaneousActivity [5907.8s, 6507.9s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/churchlandlab_ucla/Subjects/MFD_09/2023-10-19/001/alf/probe00/pykilosort/#2024-05-06#/spikes.amps.npy: 100% 647M/647M [00:06<00:00, 102MB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org/churchlandlab_ucla/Subjects/MFD_09/2023-10-19/001/alf/probe00/pykilosort/#2024-05-06#/spikes.clusters.npy: 100% 323M/323M [00:02<00:00, 110MB/s]
(S3) /root/Downloads/ONE/openalyx.internationalbrainlab.org